<div style="text-align: center;">
  <h1 style="text-align: center;">Workshop: Prototipado de Agentes con OpenAI Agents SDK</h1>
  <h3 style="text-align: center;">Master IA &amp; Data Science · EBIS Business School</h3>
  <p style="text-align: center;">Bloque A — Fundamentos del SDK</p>
</div>

## Objetivos de aprendizaje

En este notebook aprenderás:
- Qué son los agentes de IA y por qué importan
- Los 4 conceptos fundamentales del OpenAI Agents SDK: Agents, Handoffs, Guardrails, Sessions
- Cómo construir agentes con tools, validación y memoria — sobre el caso de una agencia de viajes inteligente

## Caso transversal

Construiremos **TravelMind**, un asistente de viajes que ayuda a planificar escapadas: calcula presupuestos, consulta clima y vuelos, recomienda destinos y en el Bloque B lo conectaremos con guías de viaje reales.

## Configuración

Instala las dependencias y configura el entorno. **Después de instalar, reinicia el kernel** antes de continuar.

In [ ]:
!uv pip install --system -q --upgrade openai-agents==0.4.1 nest_asyncio==1.6.0 pandas "typing_extensions>=4.13.0"
# ⚠️ Reinicia el kernel tras instalar: Kernel → Restart Kernel

In [ ]:
import nest_asyncio
nest_asyncio.apply()
print("✅ Listo.")

## 1. ¿Qué son los Agentes de IA?

Un **agente de IA** es un sistema que puede:
- Razonar sobre un problema
- Usar herramientas (tools) para obtener información o ejecutar acciones
- Tomar decisiones de forma autónoma
- Mantener contexto y memoria de conversaciones

**Diferencia clave**: Un LLM responde preguntas; un agente puede **hacer cosas** — consultar precios de vuelos, calcular presupuestos, comparar destinos, crear itinerarios personalizados.

### Por qué importa

- **Personalización**: miles de combinaciones de destino/fecha/presupuesto — un agente razona sobre el perfil del viajero.
- **Conocimiento disperso**: visados, clima, moneda, transporte — un agente lo consulta on-demand.
- **Velocidad**: el viajero no lee 50 páginas de guía; pregunta y obtiene una respuesta accionable.

## 2. OpenAI Agents SDK (2025)

El SDK oficial de OpenAI incluye 4 primitivas clave:

1. **Agents** — LLMs con instrucciones y herramientas
2. **Handoffs** — delegación entre agentes especializados
3. **Guardrails** — validación de entradas y salidas
4. **Sessions** — gestión automática del historial

### API tradicional vs Agents SDK

| Aspecto | `openai.chat.completions.create` | Agents SDK |
|---|---|---|
| Historial de mensajes | Lo mantienes tú a mano | `Session` lo gestiona |
| Esquema JSON de tools | Lo escribes tú | Generado desde la firma + docstring |
| Function calling | Detectas y ejecutas tú | Automático |
| Loop modelo ↔ tools | Lo orquestas tú | Automático |
| Delegación entre agentes | A mano | Primitiva `Handoff` |
| Validación de I/O | A mano | Primitiva `Guardrail` |

**API tradicional** = ~40 líneas de boilerplate para un agente con tools.  
**Agents SDK** = ~10 líneas para el mismo caso.

## 3. Cómo funciona un agente por dentro

Antes de escribir código, conviene tener el modelo mental claro. Cuando llamas a `Runner.run_sync()` esto es lo que ocurre:

```
Tu pregunta
    │
    ▼
┌─────────────────────────────────────────┐
│  Runner                                 │
│                                         │
│   1. Envía la pregunta al Agente        │
│   2. El Agente decide:                  │
│       ├─ ¿Necesito una tool? ──► Sí    │
│       │       └─ Ejecuta la tool        │
│       │       └─ Devuelve resultado     │
│       │       └─ Vuelve al paso 2       │
│       └─ ¿Tengo suficiente info? ─► Sí │
│               └─ Genera respuesta final │
└─────────────────────────────────────────┘
    │
    ▼
resultado.final_output
```

**`Agent`** define *quién* es el agente (instrucciones, herramientas disponibles).  
**`Runner`** es el motor que ejecuta el bucle hasta obtener una respuesta final.

## 4. Primer Agente: "Hola Mundo"

El agente más simple posible: instrucciones + una pregunta. Sin tools todavía.

In [ ]:
from agents import Agent, Runner

agente_simple = Agent(
    name="TravelMind Basic",
    instructions=(
        "Eres TravelMind, un asistente experto en viajes. "
        "Ayudas a planificar viajes, recomiendas destinos y explicas qué esperar de cada lugar. "
        "Sé concreto, práctico y entusiasta. Si no estás seguro, dilo."
    )
)

resultado = Runner.run_sync(
    agente_simple,
    "¿Qué diferencia hay entre visitar Japón en marzo vs en octubre?"
)

print(resultado.final_output)

**`final_output`** es siempre el punto de entrada al resultado — es un `str` cuando el agente no tiene `output_type` definido. En la sección 7 veremos cómo convertirlo en un objeto estructurado.

## 5. Agentes con Herramientas (Tools)

Los agentes se vuelven poderosos cuando pueden usar **herramientas**: funciones Python normales decoradas con `@function_tool`. El agente lee la descripción de la función (docstring) para decidir cuándo y cómo llamarla — no hay que decirle explícitamente "usa esta tool". 

### Ejemplo: Planificador de presupuesto de viaje

In [ ]:
from datetime import date
from agents import function_tool

@function_tool
def calcular_presupuesto_diario(
    presupuesto_total: float,
    num_dias: int,
    porcentaje_alojamiento: float = 40.0
) -> dict:
    """Distribuye un presupuesto de viaje entre alojamiento, comida y actividades.

    Args:
        presupuesto_total: Presupuesto total en euros.
        num_dias: Duración del viaje en días.
        porcentaje_alojamiento: Porcentaje destinado a alojamiento (por defecto 40%).
    """
    print("  → calcular_presupuesto_diario llamada")
    diario = presupuesto_total / num_dias
    alojamiento = diario * porcentaje_alojamiento / 100
    comida = diario * 0.35
    actividades = diario - alojamiento - comida
    return {
        "diario_total": round(diario, 2),
        "alojamiento": round(alojamiento, 2),
        "comida": round(comida, 2),
        "actividades": round(actividades, 2),
    }

@function_tool
def convertir_divisa(importe: float, moneda_origen: str, moneda_destino: str) -> float:
    """Convierte un importe entre divisas usando tasas de referencia aproximadas.

    Args:
        importe: Cantidad a convertir.
        moneda_origen: Código ISO de la moneda origen (EUR, USD, JPY, GBP, THB, MXN).
        moneda_destino: Código ISO de la moneda destino.
    """
    print(f"  → convertir_divisa {moneda_origen} → {moneda_destino}")
    tasas = {"EUR": 1.0, "USD": 1.08, "JPY": 160.0, "GBP": 0.86, "THB": 38.5, "MXN": 18.5}
    if moneda_origen not in tasas or moneda_destino not in tasas:
        return -1.0
    en_eur = importe / tasas[moneda_origen]
    return round(en_eur * tasas[moneda_destino], 2)

@function_tool
def dias_hasta_viaje(fecha_salida: str) -> int:
    """Calcula cuántos días quedan hasta la fecha de salida.

    Args:
        fecha_salida: Fecha en formato YYYY-MM-DD.
    """
    print(f"  → dias_hasta_viaje {fecha_salida}")
    salida = date.fromisoformat(fecha_salida)
    return (salida - date.today()).days

agente_presupuesto = Agent(
    name="TravelMind Presupuesto",
    instructions=(
        "Eres un planificador de viajes especializado en presupuestos. "
        "Usa las herramientas para calcular distribuciones de gasto, convertir divisas "
        "y calcular cuánto tiempo queda para preparar el viaje. Sé concreto con los números."
    ),
    tools=[calcular_presupuesto_diario, convertir_divisa, dias_hasta_viaje]
)

resultado = Runner.run_sync(
    agente_presupuesto,
    "Tengo 1200€ para 7 días en Tailandia. ¿Cómo distribuyo el presupuesto y a cuántos baht equivale lo de alojamiento?"
)

print(resultado.final_output)

Fíjate en los prints `→ tool llamada` — puedes ver exactamente qué tools eligió el agente y en qué orden. Ahora probamos con una pregunta que requiere **varias tools a la vez**:

In [ ]:
resultado = Runner.run_sync(
    agente_presupuesto,
    "Me voy a Japón el 2026-09-15 con 2000€ para 10 días. "
    "¿Cuánto me queda de presupuesto diario para actividades y a cuántos yenes equivale?"
)

print(resultado.final_output)

## 6. Tools async: dos consultas al mismo tiempo

Hasta ahora las tools son funciones normales (síncronas). Cuando una tool tiene que llamar a una API externa o una base de datos, conviene hacerla `async` — así el SDK puede ejecutar **varias tools en paralelo** en lugar de una detrás de otra.

> No hace falta entender `asyncio` en profundidad. La idea es simple: si el agente necesita consultar clima **y** vuelos al mismo tiempo, no esperamos primero uno y luego el otro — los pedimos a la vez y esperamos al más lento. El tiempo total es el máximo, no la suma.

In [ ]:
import asyncio
import time

@function_tool
async def consultar_clima(ciudad: str) -> dict:
    """Consulta el clima actual y la probabilidad de lluvia de una ciudad.

    Args:
        ciudad: Nombre de la ciudad.
    """
    await asyncio.sleep(0.8)  # simula 0.8s de latencia de una API real
    print(f"  → consultar_clima({ciudad}) terminó")
    datos = {
        "Lisboa": {"temp_c": 22, "condicion": "soleado", "lluvia_prob_pct": 10},
        "Londres": {"temp_c": 14, "condicion": "nublado", "lluvia_prob_pct": 65},
        "Bangkok": {"temp_c": 34, "condicion": "húmedo", "lluvia_prob_pct": 40},
        "Tokio":   {"temp_c": 18, "condicion": "variable", "lluvia_prob_pct": 25},
    }
    return datos.get(ciudad, {"temp_c": 20, "condicion": "variable", "lluvia_prob_pct": 30})

@function_tool
async def consultar_vuelos(origen: str, destino: str) -> dict:
    """Consulta disponibilidad y precio mínimo de vuelos directos entre dos ciudades.

    Args:
        origen: Ciudad de origen.
        destino: Ciudad de destino.
    """
    await asyncio.sleep(0.8)  # simula 0.8s de latencia
    print(f"  → consultar_vuelos({origen}→{destino}) terminó")
    rutas = {
        ("Madrid", "Lisboa"):  {"directo": True,  "precio_min_eur": 59,  "aerolineas": ["Iberia", "Vueling"]},
        ("Madrid", "Londres"): {"directo": True,  "precio_min_eur": 89,  "aerolineas": ["Iberia", "British Airways"]},
        ("Madrid", "Bangkok"): {"directo": False, "precio_min_eur": 520, "aerolineas": ["Emirates", "Qatar"]},
        ("Madrid", "Tokio"):   {"directo": False, "precio_min_eur": 680, "aerolineas": ["JAL", "ANA"]},
    }
    return rutas.get((origen, destino), {"directo": False, "precio_min_eur": 400, "aerolineas": ["varias"]})

agente_comparador = Agent(
    name="TravelMind Comparador",
    instructions=(
        "Cuando compares destinos, consulta clima y vuelos en paralelo para ahorrar tiempo. "
        "Resume la comparativa de forma clara y recomienda una opción."
    ),
    tools=[consultar_clima, consultar_vuelos],
)

t0 = time.perf_counter()
resultado = Runner.run_sync(
    agente_comparador,
    "Quiero irme este finde desde Madrid. ¿Lisboa o Londres? Dime clima y precio de vuelo de cada una."
)
print(resultado.final_output)
print(f"\n⏱  Total: {time.perf_counter() - t0:.2f}s")
print("   (Si fueran secuenciales: 0.8 + 0.8 + 0.8 + 0.8 = 3.2s. En paralelo: ~0.8s)")

## 7. Salida estructurada

Por defecto `final_output` es texto libre (un `str`). Pero podemos pedirle al agente que devuelva un **objeto Python con campos concretos** — como un formulario que el modelo tiene que rellenar.

Esto se hace con `output_type=MiClase` donde `MiClase` hereda de `BaseModel` (Pydantic). A partir de ahí `final_output` ya no es un string: es una instancia de tu clase, con acceso por atributo y validación automática.

**¿Cuándo usarlo?** Siempre que necesites procesar la respuesta del agente en código — guardarla en una base de datos, pasarla a otra función, mostrarla en una UI, etc.

In [ ]:
from typing import Literal
from pydantic import BaseModel

class RecomendacionViaje(BaseModel):
    destino: str
    tipo: Literal["playa", "ciudad", "naturaleza", "aventura", "cultural"]
    presupuesto_nivel: Literal["bajo", "medio", "alto"]
    mejor_epoca: str
    razon: str

agente_recomendador = Agent(
    name="TravelMind Recomendador",
    instructions=(
        "Recomienda el destino más adecuado para el perfil del viajero. "
        "Sé específico con el destino (ciudad o región, no solo país)."
    ),
    output_type=RecomendacionViaje,
)

resultado = Runner.run_sync(
    agente_recomendador,
    "Somos una pareja, nos encanta la historia y la gastronomía, presupuesto medio, 10 días en octubre."
)

rec = resultado.final_output  # ya no es str — es un objeto RecomendacionViaje
print(f"type(final_output) = {type(rec).__name__}")
print()
print(f"✈️  Destino:       {rec.destino}")
print(f"🏷️  Tipo:          {rec.tipo}")
print(f"💶  Presupuesto:   {rec.presupuesto_nivel}")
print(f"📅  Mejor época:   {rec.mejor_epoca}")
print(f"💬  Razón:         {rec.razon}")

## 8. Handoffs: Delegación entre Agentes

Un **handoff** es la forma de que un agente pase el control a otro más especializado. El agente coordinador analiza la consulta y decide a quién delegar — como una recepcionista que transfiere la llamada al departamento correcto.

```
Viajero → Coordinador → ¿Vuelos?     → Especialista Vuelos
                      → ¿Alojamiento? → Especialista Alojamiento  
                      → ¿Actividades? → Especialista Actividades
```

In [ ]:
from agents import handoff

agente_vuelos = Agent(
    name="Especialista en Vuelos",
    instructions=(
        "Eres experto en vuelos: rutas, aerolíneas de bajo coste, escalas, equipaje, "
        "check-in online y consejos para encontrar los mejores precios."
    )
)

agente_alojamiento = Agent(
    name="Especialista en Alojamiento",
    instructions=(
        "Eres experto en alojamiento: hoteles, hostels, Airbnb, apartamentos turísticos. "
        "Conoces las mejores zonas de cada ciudad y la relación calidad-precio."
    )
)

agente_actividades = Agent(
    name="Especialista en Actividades",
    instructions=(
        "Eres experto en experiencias de viaje: tours, museos, gastronomía local, "
        "actividades para familias, rutas de senderismo e itinerarios día a día."
    )
)

agente_coordinador = Agent(
    name="TravelMind Coordinador",
    instructions=(
        "Eres el coordinador principal de TravelMind. "
        "Analiza la consulta del viajero y delega al especialista apropiado. "
        "Si la consulta cruza dominios, delega al más relevante primero."
    ),
    handoffs=[
        handoff(
            agent=agente_vuelos,
            tool_description_override="Delega para preguntas sobre aerolíneas, rutas, equipaje o precios de vuelo"
        ),
        handoff(
            agent=agente_alojamiento,
            tool_description_override="Delega para preguntas sobre hoteles, Airbnb o dónde alojarse"
        ),
        handoff(
            agent=agente_actividades,
            tool_description_override="Delega para itinerarios, qué ver, tours o gastronomía"
        ),
    ]
)

resultado = Runner.run_sync(
    agente_coordinador,
    "¿Merece la pena pagar extra por un hotel en el centro de Roma o mejor en zona Trastevere?"
)

print(resultado.final_output)

**Ventaja clave**: cada especialista tiene un system prompt ajustado a su dominio, lo que mejora la calidad de la respuesta. Añadir un nuevo experto (visados, seguros de viaje, transporte local...) es tan simple como crear un nuevo `Agent` y añadirlo a la lista de `handoffs`.

## 9. Guardrails: Validación y Control

Los **guardrails** son filtros que se ejecutan antes (input) o después (output) del agente. Sirven para rechazar consultas inapropiadas o para garantizar que la respuesta cumple ciertas condiciones.

Piénsalos como un portero de discoteca: decide quién entra (input guardrail) y también revisa lo que sale (output guardrail).

### Paso 1 — Importaciones y tipos de datos

In [ ]:
from agents import (
    input_guardrail, output_guardrail,
    GuardrailFunctionOutput, RunContextWrapper,
    InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered,
)
from pydantic import BaseModel

class RespuestaViaje(BaseModel):
    respuesta: str
    nivel_urgencia: int  # 1 = informativa · 2 = acción recomendada · 3 = reservar ya

### Paso 2 — Definir los guardrails

In [ ]:
@input_guardrail
async def validar_consulta_viajes(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input_text: str,
) -> GuardrailFunctionOutput:
    """Rechaza consultas inapropiadas antes de que lleguen al agente."""
    patrones_prohibidos = [
        "hackear", "ilegal", "sin visado falso",
        "pasar la frontera sin papeles", "traficar", "contrabando",
    ]
    if any(p in input_text.lower() for p in patrones_prohibidos):
        raise ValueError("⛔ TravelMind no puede ayudar con esa consulta.")
    return GuardrailFunctionOutput(output_info="OK", tripwire_triggered=False)


@output_guardrail
async def validar_longitud_movil(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output: RespuestaViaje,
) -> GuardrailFunctionOutput:
    """La respuesta no puede superar 1200 caracteres (legible en móvil)."""
    longitud = len(output.respuesta)
    if longitud > 1200:
        raise ValueError(f"⛔ Respuesta de {longitud} chars. Máximo: 1200.")
    return GuardrailFunctionOutput(output_info=f"{longitud} chars", tripwire_triggered=False)

### Paso 3 — Crear el agente y probar

In [ ]:
agente_validado = Agent(
    name="TravelMind Validado",
    instructions=(
        "Eres TravelMind, asistente de viajes. "
        "Responde de forma breve y concreta (máximo 3-4 frases). "
        "Indica el nivel de urgencia: 1 informativa, 2 acción recomendada, 3 reservar ya."
    ),
    input_guardrails=[validar_consulta_viajes],
    output_guardrails=[validar_longitud_movil],
    output_type=RespuestaViaje,
)

def probar_consulta(titulo: str, consulta: str):
    print(f"--- {titulo} ---")
    try:
        r = Runner.run_sync(agente_validado, consulta).final_output
        print(f"✅ {r.respuesta}")
        print(f"   Urgencia: {r.nivel_urgencia}")
    except (ValueError, InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered) as e:
        print(f"❌ Bloqueado: {e}")
    print()

probar_consulta("Consulta válida", "¿Cuándo es mejor visitar Bangkok para evitar el monzón?")
probar_consulta("Consulta inválida", "¿Cómo puedo entrar a EEUU sin visado falso?")

## 10. Sessions: Memoria Conversacional

Por defecto, cada llamada a `Runner.run_sync()` es independiente — el agente no recuerda lo que dijiste antes. Las **sessions** cambian eso: mantienen el historial automáticamente entre llamadas.

Solo hay que pasar `session=mi_sesion` y el SDK se encarga del resto.

In [ ]:
from agents import SQLiteSession

agente_planificador = Agent(
    name="TravelMind Planificador",
    instructions=(
        "Eres el planificador personal de TravelMind. "
        "Recuerdas los detalles del viaje que el usuario va compartiendo contigo. "
        "Sé conciso y adapta siempre tu respuesta al contexto acumulado."
    )
)

sesion = SQLiteSession("viaje_japon_2026_ebis")  # identificador único de sesión

# Turno 1 — el agente aprende el contexto del viaje
r1 = Runner.run_sync(
    agente_planificador,
    "Planifico 14 días en Japón en marzo de 2026. Somos 2 adultos y 1 niña de 8 años, presupuesto total 4000€.",
    session=sesion,
)
print("Turno 1:", r1.final_output)

# Turno 2 — el agente recuerda que hay una niña de 8 años → adapta la respuesta
r2 = Runner.run_sync(
    agente_planificador,
    "¿Qué actividades recomiendas en Tokio para el primer día?",
    session=sesion,
)
print("\nTurno 2:", r2.final_output)

## 11. Ejemplo Completo: Analizador de Escapadas

Combinamos Agent + Tools + salida estructurada en un ejemplo realista: el agente analiza un dataset de destinos y recomienda el mejor según el presupuesto.

In [ ]:
import pandas as pd

opciones = pd.DataFrame({
    "destino":            ["Lisboa", "Ámsterdam", "Praga", "Dubrovnik", "Edimburgo"],
    "precio_vuelo_eur":   [89,        145,          112,     210,          130],
    "precio_hotel_noche": [95,        160,           75,     180,          110],
    "temp_media_mayo_c":  [21,         16,           19,      24,           13],
    "horas_vuelo":        [1.5,         3.0,          3.5,     2.5,          2.5],
    "valoracion_media":   [4.6,         4.4,          4.7,     4.5,          4.5],
})

@function_tool
def listar_destinos() -> list[dict]:
    """Devuelve todas las opciones de destino disponibles con precios, clima y valoraciones."""
    return opciones.to_dict(orient="records")

@function_tool
def destino_mas_barato() -> str:
    """Identifica el destino con menor coste total (vuelo ida+vuelta + 2 noches de hotel)."""
    df = opciones.copy()
    df["coste_total"] = df["precio_vuelo_eur"] * 2 + df["precio_hotel_noche"] * 2
    d = df.loc[df["coste_total"].idxmin()]
    return f"{d['destino']} con {d['coste_total']:.0f}€ totales"

@function_tool
def mejor_relacion_calidad_precio() -> dict:
    """Calcula la mejor opción ponderando valoración, precio y temperatura."""
    df = opciones.copy()
    df["score"] = (
        df["valoracion_media"] * 20
        - df["precio_vuelo_eur"] / 10
        - df["precio_hotel_noche"] / 20
        + df["temp_media_mayo_c"] / 2
    )
    d = df.loc[df["score"].idxmax()]
    return {"destino": d["destino"], "score": round(d["score"], 1), "temp_c": d["temp_media_mayo_c"]}

@function_tool
def filtrar_por_presupuesto(presupuesto_max_eur: float) -> list[dict]:
    """Filtra destinos que caben en el presupuesto máximo (vuelo ida+vuelta + 2 noches).

    Args:
        presupuesto_max_eur: Presupuesto máximo total en euros.
    """
    df = opciones.copy()
    df["coste_total"] = df["precio_vuelo_eur"] * 2 + df["precio_hotel_noche"] * 2
    validos = df[df["coste_total"] <= presupuesto_max_eur]
    return validos[["destino", "coste_total", "temp_media_mayo_c", "valoracion_media"]].to_dict(orient="records")

agente_analizador = Agent(
    name="TravelMind Analizador",
    instructions=(
        "Eres el analizador de escapadas de fin de semana de TravelMind. "
        "Usa las tools para comparar destinos y da una recomendación clara, justificada con datos."
    ),
    tools=[listar_destinos, destino_mas_barato, mejor_relacion_calidad_precio, filtrar_por_presupuesto],
)

resultado = Runner.run_sync(
    agente_analizador,
    "Tengo un finde libre en mayo, salgo desde Madrid con 400€ de presupuesto máximo. ¿A dónde me voy?"
)

print(resultado.final_output)

## 12. Más allá de los fundamentos

Cuatro características del SDK útiles en producción. Las vemos brevemente — en el Bloque B usaremos algunas de ellas.

### 12.1 Tracing automático

Cada `Runner.run_sync()` genera automáticamente una traza en `platform.openai.com/traces`: qué tools se llamaron, qué respondió el modelo en cada paso, cuánto tardó. Sin código extra — solo tienes que mirar el dashboard.

In [ ]:
import os
from agents import trace

with trace("demo workshop travelmind") as t:
    res = Runner.run_sync(
        agente_simple,
        "¿Cuáles son los 3 destinos más populares de Europa para un viaje cultural?"
    )

print(res.final_output)
print(f"\n🔗 https://platform.openai.com/traces/trace?trace_id={t.trace_id}")

### 12.2 Streaming

`Runner.run_streamed()` muestra la respuesta **token a token** mientras se genera, en lugar de esperar a que termine. Ideal para interfaces de chat.

In [ ]:
import asyncio
from openai.types.responses import ResponseTextDeltaEvent

async def demo_streaming():
    streamed = Runner.run_streamed(
        agente_simple,
        "Dame 3 consejos prácticos para viajar solo por primera vez a Asia."
    )
    async for evt in streamed.stream_events():
        if evt.type == "raw_response_event" and isinstance(evt.data, ResponseTextDeltaEvent):
            print(evt.data.delta, end="", flush=True)
    print()

asyncio.run(demo_streaming())

### 12.3 Context tipado

En producción necesitas pasar datos del usuario a las tools (su ID, sus preferencias guardadas, una conexión a base de datos...) sin que el modelo los vea directamente. El patrón es:

1. Defines un `dataclass` con tus datos.
2. Lo pasas a `Runner.run_sync(..., context=mis_datos)`.
3. Las tools lo reciben como primer parámetro (`RunContextWrapper`).

**Caso de uso típico**: personalizar la respuesta según el historial del usuario autenticado, sin exponer ese historial en el prompt.

In [ ]:
from dataclasses import dataclass
from agents import RunContextWrapper

@dataclass
class PerfilViajero:
    user_id: str
    nombre: str
    viajes_anteriores: list[str]

@function_tool
def historial_viajero(ctx: RunContextWrapper[PerfilViajero]) -> dict:
    """Devuelve el historial de viajes del usuario actual."""
    p = ctx.context
    return {"nombre": p.nombre, "viajes": p.viajes_anteriores, "total": len(p.viajes_anteriores)}

agente_personalizado = Agent[PerfilViajero](
    name="TravelMind Personalizado",
    instructions=(
        "Consulta el historial del viajero antes de recomendar. "
        "No repitas destinos que ya haya visitado. Personaliza por nombre."
    ),
    tools=[historial_viajero],
)

perfil = PerfilViajero(user_id="U-1042", nombre="María", viajes_anteriores=["París", "Roma", "Berlín", "Lisboa"])

resultado = Runner.run_sync(
    agente_personalizado,
    "¿Qué ciudad europea me recomiendas para el próximo puente?",
    context=perfil,
)
print(resultado.final_output)

### 12.4 Hosted tools y MCP

Además de tus propias funciones Python, el SDK da acceso a **tools alojadas en la infraestructura de OpenAI** — no necesitas implementarlas tú:

- `WebSearchTool()` — búsqueda web en tiempo real
- `FileSearchTool(...)` — RAG sobre documentos subidos a OpenAI
- `CodeInterpreterTool()` — ejecutar Python en un sandbox

Y con **MCP** (Model Context Protocol) puedes conectar el agente a cualquier servidor externo de tools — bases de datos, APIs internas, sistemas legacy — usando un estándar abierto. En el Bloque B usaremos esta idea para conectar el agente con las guías de viaje.

In [ ]:
from agents import WebSearchTool

agente_web = Agent(
    name="TravelMind Web",
    instructions=(
        "Usa búsqueda web para responder preguntas que requieran información actualizada. "
        "Cita siempre la fuente (dominio) al final de tu respuesta."
    ),
    tools=[WebSearchTool()],
)

resultado = Runner.run_sync(
    agente_web,
    "¿Necesitan visado los ciudadanos españoles para entrar a Japón en 2025?"
)
print(resultado.final_output)

## 13. Ejemplo integrador: TravelMind Assistant

Hasta aquí hemos visto cada pieza por separado. Ahora las juntamos todas en un sistema real.

**Lo que vamos a construir**: un asistente de viajes con arquitectura multi-agente que:

1. **Valida** la consulta antes de procesarla (guardrail de entrada)
2. **Consulta datos** con tools (vuelos y hoteles con precios mock)
3. **Delega** a especialistas según el tema (handoffs)
4. **Recuerda** el contexto entre turnos (session)

```
Viajero
   |
   v
[Input Guardrail] -- consulta fuera de viajes --> rechazada
   |
   v
[Coordinador TravelMind]
   |-- tool: buscar_vuelos()
   |-- tool: buscar_hoteles()
   |-- handoff --> [Especialista Visados]
   +-- handoff --> [Especialista Actividades]
   |
   v
[SQLiteSession] -- recuerda todo el hilo
```

Ejecuta las celdas una a una y observa en los prints qué está pasando en cada momento.

### Paso 1 — Tools: vuelos y hoteles

Dos funciones con datos mock que simulan consultas a una API real.
El agente las llamará automáticamente cuando la pregunta lo requiera — sin que se lo digamos explícitamente.

In [ ]:
from agents import Agent, Runner, function_tool, handoff, input_guardrail, GuardrailFunctionOutput, RunContextWrapper, SQLiteSession

# Datos mock — en producción serían llamadas a una API real
VUELOS = {
    ("Madrid", "Tokyo"):    {"precio": 780, "aerolinea": "Iberia/JAL",    "duracion_h": 14},
    ("Madrid", "Bangkok"):  {"precio": 520, "aerolinea": "Qatar Airways",  "duracion_h": 11},
    ("Madrid", "Lisboa"):   {"precio":  65, "aerolinea": "Iberia Express", "duracion_h":  1},
    ("Madrid", "Tokio"):    {"precio": 780, "aerolinea": "Iberia/JAL",    "duracion_h": 14},
}

HOTELES = {
    "Tokyo":   [{"nombre": "APA Hotel Shinjuku", "precio_noche":  85, "estrellas": 3},
                {"nombre": "Park Hyatt Tokyo",   "precio_noche": 420, "estrellas": 5}],
    "Bangkok": [{"nombre": "Lub d Silom",        "precio_noche":  28, "estrellas": 2},
                {"nombre": "Mandarin Oriental",  "precio_noche": 380, "estrellas": 5}],
    "Lisboa":  [{"nombre": "Yes! Lisbon Hostel", "precio_noche":  22, "estrellas": 1},
                {"nombre": "Bairro Alto Hotel",  "precio_noche": 310, "estrellas": 5}],
}

@function_tool
def buscar_vuelos(origen: str, destino: str) -> dict:
    """Busca vuelos disponibles entre dos ciudades y devuelve precio y aerolinea.

    Args:
        origen: Ciudad de origen del vuelo.
        destino: Ciudad de destino del vuelo.
    """
    print(f"  [tool] buscar_vuelos({origen} -> {destino})")
    clave = (origen.capitalize(), destino.capitalize())
    if clave in VUELOS:
        return VUELOS[clave]
    return {"error": f"No hay vuelos directos de {origen} a {destino} en nuestra base de datos."}

@function_tool
def buscar_hoteles(ciudad: str, max_precio_noche: float = 9999) -> list:
    """Busca hoteles en una ciudad filtrando por precio maximo por noche.

    Args:
        ciudad: Ciudad donde buscar alojamiento.
        max_precio_noche: Precio maximo por noche en euros (opcional).
    """
    print(f"  [tool] buscar_hoteles({ciudad}, max={max_precio_noche} EUR/noche)")
    ciudad_cap = ciudad.capitalize()
    hoteles = HOTELES.get(ciudad_cap, [])
    filtrados = [h for h in hoteles if h["precio_noche"] <= max_precio_noche]
    if not filtrados:
        return [{"mensaje": f"No hay hoteles en {ciudad} dentro de ese presupuesto."}]
    return filtrados

print("Tools definidas: buscar_vuelos, buscar_hoteles")

### Paso 2 — Especialistas con handoff

Dos agentes especializados. El coordinador les cederá el control cuando detecte
que la pregunta encaja con su dominio — sin lógica de enrutamiento manual.

In [ ]:
especialista_visados = Agent(
    name="Especialista en Visados",
    instructions=(
        "Eres experto en visados y documentacion de viaje para ciudadanos espanoles. "
        "Explica con precision los requisitos de entrada, duracion de estancia permitida "
        "y cualquier tramite previo necesario. Si no tienes datos actualizados, recomienda "
        "verificar en la embajada correspondiente."
    )
)

especialista_actividades = Agent(
    name="Especialista en Actividades",
    instructions=(
        "Eres experto en planes, excursiones y experiencias de viaje. "
        "Recomienda actividades concretas adaptadas al perfil del viajero (familia, mochilero, lujo...). "
        "Incluye siempre una estimacion de precio y la duracion aproximada de cada actividad."
    )
)

print("Especialistas definidos: visados y actividades")

### Paso 3 — Guardrail: solo consultas de viajes

Antes de que cualquier mensaje llegue al coordinador, el guardrail comprueba
que la consulta tiene que ver con viajes. Si no, la corta de raiz — sin gastar tokens.

In [ ]:
@input_guardrail
async def solo_viajes(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input_text: str,
) -> GuardrailFunctionOutput:
    """Rechaza consultas que no tengan relacion con viajes o turismo."""
    temas_ajenos = ["receta", "futbol", "bolsa", "codigo python", "matematicas", "politica"]
    texto = input_text.lower()
    if any(tema in texto for tema in temas_ajenos):
        raise ValueError("TravelMind solo responde consultas relacionadas con viajes.")
    return GuardrailFunctionOutput(output_info="OK", tripwire_triggered=False)

print("Guardrail definido: solo_viajes")

### Paso 4 — Coordinador: el agente central

El coordinador recibe todas las consultas. Usa las tools para datos concretos
y hace handoff a los especialistas cuando el tema lo requiere.
El guardrail se aplica aqui, antes de que la consulta llegue al modelo.

In [ ]:
coordinador = Agent(
    name="TravelMind Coordinador",
    instructions=(
        "Eres TravelMind, el asistente central de una agencia de viajes inteligente. "
        "Tu trabajo es dar respuestas completas y practicas. Tienes estas capacidades:\n"
        "- Usa buscar_vuelos() cuando te pregunten por precios o disponibilidad de vuelos.\n"
        "- Usa buscar_hoteles() cuando te pregunten por alojamiento.\n"
        "- Delega al Especialista en Visados para requisitos de entrada y documentacion.\n"
        "- Delega al Especialista en Actividades para planes, excursiones o que hacer.\n"
        "Se concreto y usa los datos que devuelven las tools — no inventes precios."
    ),
    tools=[buscar_vuelos, buscar_hoteles],
    handoffs=[
        handoff(especialista_visados),
        handoff(especialista_actividades),
    ],
    input_guardrails=[solo_viajes],
)

print("Coordinador listo: tools + handoffs + guardrail")

### Paso 5 — Primera consulta: vuelo + hotel a la vez

La pregunta necesita dos tools. Observa en los prints que el agente las llama
en paralelo — no espera a una para lanzar la otra.

In [ ]:
from agents.exceptions import InputGuardrailTripwireTriggered

sesion = SQLiteSession("travelmind_demo_clase")

def preguntar(consulta: str):
    """Envia una consulta al coordinador dentro de la sesion activa."""
    print(f"\nUsuario: {consulta}")
    print("-" * 55)
    try:
        resultado = Runner.run_sync(coordinador, consulta, session=sesion)
        print(resultado.final_output)
    except (InputGuardrailTripwireTriggered, ValueError) as e:
        print(f"[Bloqueado] {e}")

# Necesita buscar_vuelos Y buscar_hoteles — fjate cuantas tools llama
preguntar("Cuanto cuesta el vuelo Madrid-Tokio y que hoteles hay por menos de 150 EUR la noche?")

### Paso 6 — El coordinador delega a un especialista

Ahora preguntamos algo fuera del dominio del coordinador.
Detecta que encaja con visados y hace un handoff — el Especialista en Visados responde.

In [ ]:
# El coordinador no sabe de visados en detalle -> handoff al especialista
preguntar("Necesito visado para entrar a Japon con pasaporte espanol?")

### Paso 7 — La sesion recuerda el contexto

Preguntamos algo vago que solo tiene sentido si se recuerdan los turnos anteriores.
Gracias a SQLiteSession el agente sabe que estamos hablando de Japon y Tokio.

In [ ]:
# Sin sesion esta pregunta seria ambigua — con sesion el agente lo entiende
preguntar("Y que actividades me recomiendas para esa ciudad?")

### Paso 8 — El guardrail en accion

Enviamos una consulta que no tiene nada que ver con viajes.
El guardrail la para antes de que llegue al modelo: cero tokens gastados, respuesta instantanea.

In [ ]:
# Fuera del ambito -> el guardrail lo bloquea sin llamar al modelo
preguntar("Cual es la capital de Francia y como se calcula el area de un triangulo?")

### Que hemos visto en este ejemplo

| Pieza | Donde aparece |
|---|---|
| **Agent** | Coordinador + 2 especialistas |
| **function_tool** | `buscar_vuelos`, `buscar_hoteles` |
| **handoff** | Coordinador delega a Visados / Actividades |
| **input_guardrail** | `solo_viajes` filtra consultas ajenas |
| **SQLiteSession** | Recuerda el hilo entre los 4 turnos |

Este es exactamente el patron que usaras en el **Reto del Asistente de Guias de Viaje** —
pero con RAG real sobre PDFs en lugar de datos mock.

## Resumen

| Concepto | En una frase | Cuándo usarlo |
|---|---|---|
| **Agent** | LLM con instrucciones y herramientas | Siempre — es la base |
| **Tools** | Funciones Python que el agente puede invocar | Cuando necesitas que el agente *haga* algo |
| **Handoffs** | Un agente pasa el control a otro más especializado | Sistemas multi-agente |
| **Guardrails** | Filtros de entrada y salida | Control de calidad y seguridad |
| **Sessions** | Historial conversacional automático | Diálogos multi-turno |
| **output_type** | La respuesta es un objeto tipado, no texto libre | Cuando procesas la respuesta en código |

## Ejercicios

A partir de aquí usa un asistente de código si quieres: **Claude Code · Cursor · GitHub Copilot**.

Regla única: revisa siempre el código generado antes de ejecutarlo.

### Ejercicio 1 — Conversor de unidades de viaje (Básico)

Crea un agente con al menos 3 tools de conversión útiles para viajeros: km ↔ millas, kg ↔ libras (equipaje), °C ↔ °F (clima), diferencia horaria, etc.

Pruébalo con al menos 2 conversiones distintas en la misma pregunta.

In [ ]:
# TU CÓDIGO AQUÍ

# Esqueleto de partida:
# @function_tool
# def celsius_a_fahrenheit(temp_c: float) -> float:
#     """Convierte temperatura de grados Celsius a Fahrenheit.
#     Args:
#         temp_c: Temperatura en grados Celsius.
#     """
#     ...
#
# agente_conversor = Agent(
#     name="TravelMind Conversor",
#     instructions="...",
#     tools=[celsius_a_fahrenheit, ...]
# )
#
# Runner.run_sync(agente_conversor, "Bangkok está a 34°C y la maleta pesa 23kg. ¿En Fahrenheit y libras?")

### Ejercicio 2 — Sistema multi-especialista (Intermedio)

Crea un sistema con un coordinador y 3 especialistas (Vuelos, Alojamiento, Actividades) usando handoffs. Añade una tool `consultar_perfil_viajero(user_id)` en el coordinador que devuelva datos mock del usuario antes de delegar.

Prueba con 3 preguntas que toquen dominios distintos.

In [ ]:
# TU CÓDIGO AQUÍ

# Esqueleto de partida:
# @function_tool
# def consultar_perfil_viajero(user_id: str) -> dict:
#     """Devuelve el perfil del viajero (mock).
#     Args:
#         user_id: Identificador del usuario.
#     """
#     perfiles = {
#         "U-001": {"nombre": "Carlos", "preferencia": "económico", "viajes": ["París", "Berlín"]},
#     }
#     return perfiles.get(user_id, {"nombre": "Desconocido", "preferencia": "medio", "viajes": []})
#
# especialista_vuelos   = Agent(name="...", instructions="...")
# especialista_hotel    = Agent(name="...", instructions="...")
# especialista_activ    = Agent(name="...", instructions="...")
#
# coordinador = Agent(
#     name="...",
#     instructions="...",
#     tools=[consultar_perfil_viajero],
#     handoffs=[handoff(agent=especialista_vuelos, ...), ...]
# )

### Ejercicio 3 — Planificador con guardrails + sesión (Avanzado)

Crea un agente planificador que:
- Tenga al menos 2 tools (`calcular_coste_total`, `recomendar_seguro` o similar)
- Un input guardrail que rechace preguntas fuera del ámbito de viajes
- Un output guardrail que verifique que la respuesta incluye el coste estimado
- Use `SQLiteSession` para recordar el contexto entre al menos 3 turnos

**Bonus**: añade una tool `generar_checklist_viaje(destino, dias, viajeros)` que devuelva una lista de preparativos.

In [ ]:
# TU CÓDIGO AQUÍ

# Esqueleto de partida:
# @function_tool
# def calcular_coste_total(precio_vuelo: float, precio_hotel_noche: float, num_noches: int) -> dict:
#     """Calcula el coste total de un viaje sumando vuelo y alojamiento.
#     Args:
#         precio_vuelo: Precio del vuelo ida y vuelta en euros.
#         precio_hotel_noche: Precio por noche de hotel en euros.
#         num_noches: Número de noches de alojamiento.
#     """
#     ...
#
# @input_guardrail
# async def solo_viajes(ctx, agent, input_text):
#     ...
#
# sesion = SQLiteSession("planificacion_001")
# # Turno 1: contexto inicial
# # Turno 2: primera consulta
# # Turno 3: el agente recuerda el contexto

## Recursos

- **OpenAI Agents SDK**: https://openai.github.io/openai-agents-python/
- **Repo GitHub**: https://github.com/openai/openai-agents-python
- **Pydantic AI** (alternativa): https://ai.pydantic.dev
- **Anthropic — Building effective agents**: https://www.anthropic.com/research/building-effective-agents